# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
# YOUR CODE HERE
!pip install -q -U bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # must match NB2 exactly

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit (this may take a minute)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("")
print("Model loaded successfully.")
print("Model device:", next(model.parameters()).device)
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
print("GPU memory reserved (GB):", round(torch.cuda.memory_reserved() / 1e9, 2))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.1 MB/s eta 0:00:00
Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully.
Model device: cuda:0
GPU memory allocated (GB): 1.15
GPU memory reserved (GB): 1.2


In [3]:
# YOUR CODE HERE
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
df_test = pd.read_csv(DRIVE_DIR + "/df_test.csv")

print("df_test loaded:", df_test.shape)
print(df_test.head(3))
import json
import time

def generate_baseline_response(prompt_text, model, tokenizer, max_new_tokens=120):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated_text.strip()


results = []
start_time = time.time()

for idx in range(len(df_test)):
    test_query = df_test["text"].iloc[idx]
    ground_truth = df_test["json_target"].iloc[idx]
    chatml_prompt = df_test["chatml_prompt"].iloc[idx]

    baseline_output = generate_baseline_response(chatml_prompt, model, tokenizer)

    results.append({
        "test_query": test_query,
        "ground_truth": ground_truth,
        "baseline_output": baseline_output
    })

    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_test) - (idx + 1)) / rate
        print("Processed", idx + 1, "/", len(df_test), "- elapsed:", round(elapsed, 1), "s - est. remaining:", round(remaining, 1), "s")

total_time = time.time() - start_time
print("")
print("Baseline generation complete.")
print("Total rows:", len(results))
print("Total time:", round(total_time, 1), "seconds")
print("Avg time per row:", round(total_time / len(results), 3), "seconds")

print("")
print("--- Sample outputs ---")
for r in results[:3]:
    print("Query:", r["test_query"])
    print("Ground truth:", r["ground_truth"])
    print("Baseline output:", r["baseline_output"])
    print("")

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
outputs_path = DRIVE_DIR + "/outputs.json"

with open(outputs_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved outputs.json to:", outputs_path)
print("Also saving a local copy for immediate reuse in this session")
with open("outputs.json", "w") as f:
    json.dump(results, f, indent=2)

Mounted at /content/drive
df_test loaded: (391, 5)
                                                text                   intent  \
0               need help to shop several of ur item              place_order   
1  i need assistance trying to update the shippin...  change_shipping_address   
2  I do not know how to check the cancellation  p...   check_cancellation_fee   

   category                                        json_target  \
0     ORDER        {"intent":"place_order","category":"ORDER"}   
1  SHIPPING  {"intent":"change_shipping_address","category"...   
2    CANCEL  {"intent":"check_cancellation_fee","category":...   

                                       chatml_prompt  
0  <|im_start|>system\nYou are a customer support...  
1  <|im_start|>system\nYou are a customer support...  
2  <|im_start|>system\nYou are a customer support...  
Processed 50 / 391 - elapsed: 104.3 s - est. remaining: 711.6 s
Processed 100 / 391 - elapsed: 182.0 s - est. remaining: 529.7 s
Processed 

#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [4]:
# YOUR CODE HERE
FREE_TEXT_SYSTEM_PROMPT = (
    "You are a helpful customer support assistant. Answer the customer's "
    "question directly and concisely based on company policy."
)

def build_free_text_prompt(query, tokenizer):
    messages = [
        {"role": "system", "content": FREE_TEXT_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Focus on shipping/delivery-related test queries, since the known
# ground-truth rule ("3-7 business days") lives in shipping_delays.md
shipping_intents = ["delivery_options", "track_order", "shipping_delays", "check_shipping_address"]
shipping_queries = df_test[df_test["intent"].isin(shipping_intents)].reset_index(drop=True)

print("Shipping-related test queries found:", len(shipping_queries))
print(shipping_queries["intent"].value_counts())

hallucination_log = []

for idx in range(len(shipping_queries)):
    query = shipping_queries["text"].iloc[idx]
    prompt = build_free_text_prompt(query, tokenizer)
    response = generate_baseline_response(prompt, model, tokenizer, max_new_tokens=150)

    hallucination_log.append({
        "query": query,
        "intent": shipping_queries["intent"].iloc[idx],
        "baseline_response": response
    })

print("")
print("--- All shipping-query baseline responses ---")
for i, entry in enumerate(hallucination_log):
    print("")
    print(str(i) + ". Query:", entry["query"])
    print("   Intent:", entry["intent"])
    print("   Response:", entry["baseline_response"])

Shipping-related test queries found: 28
intent
delivery_options    15
track_order         13
Name: count, dtype: int64

--- All shipping-query baseline responses ---

0. Query: I do not know how to see the status of order {{Order Number}}
   Intent: track_order
   Response: To check the status of your order, please follow these steps:

1. Log in to your account.
2. Go to the "My Orders" section.
3. Locate the order with the number {{Order Number}}.
4. Click on it to view its details.

If you're still unable to find it, try refreshing the page or clearing your browser cache. If issues persist, contact our support team for assistance.

1. Query: wanna see what shipment methods i can choose how do i do it
   Intent: delivery_options
   Response: To view your available shipping options, please visit our website and navigate to the "My Account" section. From there, select "View My Orders" or "Track My Order," depending on which information you're looking for. If you have any specific questi

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [5]:
# YOUR CODE HERE
import json
import os

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Verify 'results' exists (built during the full 391-row baseline
# generation loop) before saving
# ------------------------------------------------------------
assert "results" in dir(), "results not found - run the full baseline generation loop first"
assert len(results) == len(df_test), "results size (" + str(len(results)) + ") does not match df_test (" + str(len(df_test)) + ")"

outputs_path = DRIVE_DIR + "/outputs.json"

with open(outputs_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved outputs.json to Drive:", outputs_path)
print("Total records:", len(results))

# Local copy too, for immediate reuse without remounting Drive
with open("outputs.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved local copy: outputs.json")

# ------------------------------------------------------------
# Quick integrity check on the saved file
# ------------------------------------------------------------
with open(outputs_path, "r") as f:
    reloaded = json.load(f)

assert len(reloaded) == len(results), "Reload size mismatch!"
assert set(reloaded[0].keys()) == {"test_query", "ground_truth", "baseline_output"}, "Schema mismatch!"

print("")
print("Integrity check PASSED - outputs.json contains", len(reloaded), "records with correct schema.")
print("NB4, NB5, NB7 can now load this via json.load(open('outputs.json'))")
hallucination_path = DRIVE_DIR + "/baseline_hallucination_analysis.json"
with open(hallucination_path, "w") as f:
    json.dump(hallucination_log, f, indent=2)
print("Saved hallucination analysis to:", hallucination_path)

Saved outputs.json to Drive: /content/drive/MyDrive/customer_support_llm_project/artifacts/outputs.json
Total records: 391
Saved local copy: outputs.json

Integrity check PASSED - outputs.json contains 391 records with correct schema.
NB4, NB5, NB7 can now load this via json.load(open('outputs.json'))
Saved hallucination analysis to: /content/drive/MyDrive/customer_support_llm_project/artifacts/baseline_hallucination_analysis.json


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**